In [1]:
# 1. ENVIRONMENT SETUP (Run this cell first in Colab)
!pip uninstall -y unsloth unsloth_zoo
!pip install --prefer-binary "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps --prefer-binary xformers trl peft accelerate bitsandbytes
!pip install openai datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-69x1i0g2/unsloth_d30ee286ff0d41ca809d68980b92c795
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-69x1i0g2/unsloth_d30ee286ff0d41ca809d68980b92c795
  Resolved https://github.com/unslothai/unsloth.git to commit 040b80a60e6bf0278c0b81ee986cf9b87b9d4f6d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 135.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.9 MB/s eta 0:00:0

In [3]:
# 2. THE FLYWHEEL SCRIPT (Run this in the next Colab cell)
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from openai import OpenAI
import json
import time

# --- CONFIGURATION ---
MODEL_ID = "nallaramu/deliberate-qwen-2.5-3b-reasoning"
GROQ_API_KEY = "YOUR_GROQ_API_KEY" # Insert your Groq key here
OUTPUT_FILE = "dpo_preference_pairs.jsonl"
NUM_SAMPLES = 200 # 200 pairs is enough for a strong DPO signal on a 3B model

# Initialize Groq Client
client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")

# Load Student Model for Generation
print(f"Loading {MODEL_ID}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

# Load Unseen Data (GSM8K Test Split)
dataset = load_dataset("openai/gsm8k", "main", split="test")

# --- THE JUDGE PROMPT ---
JUDGE_SYSTEM_PROMPT = """You are an expert AI Alignment Judge.
You will be given a math question, the Ground Truth answer, and two reasoning paths (Response A and Response B).
Your job is to pick the BEST response based on:
1. Logical correctness (Does it reach the Ground Truth?).
2. Conciseness (Is it direct and free of repetitive rambling?).
3. Formatting (Does it properly use <think> and </answer> tags without trailing garbage?).

Output ONLY the letter "A" or "B". Do not output any other text."""

def generate_two_paths(question):
    prompt = f"### Question:\n{question}\n\n### Reasoning:\n"
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # We use do_sample=True and temperature=0.8 to force the model to think differently each time
    paths =[]
    for _ in range(2):
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            use_cache=True,
            do_sample=True,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id
        )
        new_tokens = outputs[0][inputs.input_ids.shape[1]:]
        paths.append(tokenizer.decode(new_tokens, skip_special_tokens=False))
    return paths[0], paths[1]

def judge_responses(question, gt, resp_a, resp_b):
    user_content = f"Question: {question}\nGround Truth: {gt}\n\n--- Response A ---\n{resp_a}\n\n--- Response B ---\n{resp_b}"
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user", "content": user_content}
            ],
            temperature=0.0,
            max_tokens=5
        )
        winner = response.choices[0].message.content.strip().upper()
        return winner if winner in ["A", "B"] else None
    except Exception as e:
        print(f"Groq API Error: {e}")
        return None

# --- THE FLYWHEEL LOOP ---
print("\nStarting the Data Flywheel...")
preference_data =[]
success_count = 0

for i, item in enumerate(dataset):
    if success_count >= NUM_SAMPLES:
        break

    question = item['question']
    gt_answer = item['answer'].split("####")[-1].strip()

    print(f"\n[{success_count+1}/{NUM_SAMPLES}] Processing question...")

    # 1. Generate candidate paths
    path_a, path_b = generate_two_paths(question)

    # 2. Ask the Judge
    winner = judge_responses(question, gt_answer, path_a, path_b)
    time.sleep(2) # Respect API rate limits

    if not winner:
        continue

    chosen = path_a if winner == "A" else path_b
    rejected = path_b if winner == "A" else path_a

    # 3. LENGTH-DEBIASING FILTER (The "Expert Signal")
    # We only keep the pair if the Chosen response is shorter or equal to the Rejected one
    chosen_len = len(tokenizer.encode(chosen))
    rejected_len = len(tokenizer.encode(rejected))

    if chosen_len > rejected_len:
        print(f"   [Filtered] Chosen path was longer ({chosen_len} vs {rejected_len} tokens). Discarding to prevent DPO length bias.")
        continue

    print(f"   [Kept] Chosen: {chosen_len} tokens | Rejected: {rejected_len} tokens.")

    # 4. Format for Hugging Face DPOTrainer
    # DPO requires the exact prompt that preceded the chosen/rejected texts
    dpo_prompt = f"### Question:\n{question}\n\n### Reasoning:\n"

    pair = {
        "prompt": dpo_prompt,
        "chosen": chosen,
        "rejected": rejected
    }

    # Save incrementally
    with open(OUTPUT_FILE, "a") as f:
        f.write(json.dumps(pair) + "\n")

    preference_data.append(pair)
    success_count += 1

print(f"\n🎉 Flywheel Complete! Saved {len(preference_data)} length-debiased preference pairs to {OUTPUT_FILE}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading nallaramu/deliberate-qwen-2.5-3b-reasoning...
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Groq API Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqm5p3mxezj9d0d71brp6a03` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99618, Requested 780. Please try again in 5m43.872s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[75/200] Processing question...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 